In [5]:
import pandas as pd
import numpy as np

evaluation_df = pd.read_csv(
    "../data/processed/taxonomy_agent_evaluation.csv"
)

print("Evaluation shape:", evaluation_df.shape)

display(evaluation_df)

Evaluation shape: (5, 7)


,test_id,customer_message,predicted_intent,intent_confidence,retrieval_similarity,decision,final_response
0,1,My battery is draining very quickly after upda...,battery_power,0.404,0.882,use_retrieved_evidence,"Thanks for reaching out. That's unexpected, bu..."
1,2,My keyboard keeps changing the word I to somet...,keyboard_input,0.514,0.825,use_retrieved_evidence,Could you tell us which iPhone model and iOS v...
2,3,I cannot log in to my Apple ID.,apple_id_icloud,0.195,0.770,clarify_or_escalate,Could you tell us what happens when you try to...
3,4,My iPhone keeps freezing after the latest update.,performance_freezing,0.249,0.860,use_retrieved_evidence,Could you tell us which iPhone model and iOS v...
4,5,I have a problem with my phone.,device_hardware,0.153,0.796,clarify_or_escalate,Could you tell us which device you're using an...


In [6]:
print("Number of test cases:", len(evaluation_df))

print(
    "Average intent confidence:",
    round(
        evaluation_df["intent_confidence"].mean(),
        3
    )
)

print(
    "Average retrieval similarity:",
    round(
        evaluation_df["retrieval_similarity"].mean(),
        3
    )
)

Number of test cases: 5
Average intent confidence: 0.303
Average retrieval similarity: 0.827


In [7]:
decision_counts = (
    evaluation_df["decision"]
    .value_counts()
)

display(decision_counts)

decision
use_retrieved_evidence    3
clarify_or_escalate       2
Name: count, dtype: int64

In [8]:
confidence_summary = evaluation_df[
    [
        "test_id",
        "predicted_intent",
        "intent_confidence"
    ]
].sort_values(
    "intent_confidence"
)

display(confidence_summary)

,test_id,predicted_intent,intent_confidence
4,5,device_hardware,0.153
2,3,apple_id_icloud,0.195
3,4,performance_freezing,0.249
0,1,battery_power,0.404
1,2,keyboard_input,0.514


In [9]:
retrieval_summary = evaluation_df[
    [
        "test_id",
        "predicted_intent",
        "retrieval_similarity"
    ]
].sort_values(
    "retrieval_similarity"
)

display(retrieval_summary)

,test_id,predicted_intent,retrieval_similarity
2,3,apple_id_icloud,0.770
4,5,device_hardware,0.796
1,2,keyboard_input,0.825
3,4,performance_freezing,0.860
0,1,battery_power,0.882


In [10]:
low_confidence = evaluation_df[
    evaluation_df["intent_confidence"] < 0.20
]

print(
    "Low-confidence cases:",
    len(low_confidence)
)

display(low_confidence)

Low-confidence cases: 2


,test_id,customer_message,predicted_intent,intent_confidence,retrieval_similarity,decision,final_response
2,3,I cannot log in to my Apple ID.,apple_id_icloud,0.195,0.770,clarify_or_escalate,Could you tell us what happens when you try to...
4,5,I have a problem with my phone.,device_hardware,0.153,0.796,clarify_or_escalate,Could you tell us which device you're using an...


In [11]:
display(
    evaluation_df[
        [
            "customer_message",
            "predicted_intent",
            "intent_confidence",
            "retrieval_similarity",
            "decision"
        ]
    ]
)

,customer_message,predicted_intent,intent_confidence,retrieval_similarity,decision
0,My battery is draining very quickly after upda...,battery_power,0.404,0.882,use_retrieved_evidence
1,My keyboard keeps changing the word I to somet...,keyboard_input,0.514,0.825,use_retrieved_evidence
2,I cannot log in to my Apple ID.,apple_id_icloud,0.195,0.770,clarify_or_escalate
3,My iPhone keeps freezing after the latest update.,performance_freezing,0.249,0.860,use_retrieved_evidence
4,I have a problem with my phone.,device_hardware,0.153,0.796,clarify_or_escalate


In [13]:
failure_analysis = evaluation_df.copy()

failure_analysis["confidence_level"] = np.where(
    failure_analysis["intent_confidence"] < 0.20,
    "Low confidence",
    "Acceptable confidence"
)

failure_analysis["retrieval_level"] = np.where(
    failure_analysis["retrieval_similarity"] >= 0.80,
    "Strong retrieval",
    "Weaker retrieval"
)

display(
    failure_analysis[
        [
            "customer_message",
            "predicted_intent",
            "intent_confidence",
            "retrieval_similarity",
            "confidence_level",
            "retrieval_level",
            "decision"
        ]
    ]
)

,customer_message,predicted_intent,intent_confidence,retrieval_similarity,confidence_level,retrieval_level,decision
0,My battery is draining very quickly after upda...,battery_power,0.404,0.882,Acceptable confidence,Strong retrieval,use_retrieved_evidence
1,My keyboard keeps changing the word I to somet...,keyboard_input,0.514,0.825,Acceptable confidence,Strong retrieval,use_retrieved_evidence
2,I cannot log in to my Apple ID.,apple_id_icloud,0.195,0.770,Low confidence,Weaker retrieval,clarify_or_escalate
3,My iPhone keeps freezing after the latest update.,performance_freezing,0.249,0.860,Acceptable confidence,Strong retrieval,use_retrieved_evidence
4,I have a problem with my phone.,device_hardware,0.153,0.796,Low confidence,Weaker retrieval,clarify_or_escalate


In [14]:
decision_summary = (
    evaluation_df
    .groupby("decision")
    .agg(
        cases=("test_id", "count"),
        avg_intent_confidence=("intent_confidence", "mean"),
        avg_retrieval_similarity=("retrieval_similarity", "mean")
    )
    .reset_index()
)

decision_summary["avg_intent_confidence"] = (
    decision_summary["avg_intent_confidence"].round(3)
)

decision_summary["avg_retrieval_similarity"] = (
    decision_summary["avg_retrieval_similarity"].round(3)
)

display(decision_summary)

,decision,cases,avg_intent_confidence,avg_retrieval_similarity
0,clarify_or_escalate,2,0.174,0.783
1,use_retrieved_evidence,3,0.389,0.856
